# Structured outputs and middleware

This notebook asks a model for validated Python data instead of free-form text, then uses middleware to summarise a long conversation before its history grows too large.


In [ ]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from pydantic import BaseModel, Field

# Load GROQ_API_KEY from the local .env file.
load_dotenv()

groq_model = init_chat_model(
    "openai/gpt-oss-120b",
    model_provider="groq",
)


## Ask for a structure instead of prose

A Pydantic model describes the data our application expects. LangChain uses this schema when it asks the chat model for a structured response, then validates the result before returning it to our code.


In [ ]:
class LessonRecommendation(BaseModel):
    """The structured result our course-planning code expects."""

    lesson_number: int = Field(description="The recommended lesson number.")
    topic: str = Field(description="The main topic of the lesson.")
    reason: str = Field(description="Why this lesson is the next useful step.")
    preparation: list[str] = Field(description="Short preparation steps for the learner.")


# Wrap the chat model so `invoke` returns a LessonRecommendation instance.
structured_groq_model = groq_model.with_structured_output(LessonRecommendation)


In [ ]:
recommendation = structured_groq_model.invoke(
    "A learner has made one direct LLM call and wants to switch providers safely. "
    "Recommend the next lesson from this course."
)

print(recommendation)
print("Topic:", recommendation.topic)
print("First preparation step:", recommendation.preparation[0])


## Summarise long conversation history with middleware

Conversation history is useful until it becomes too large. `SummarizationMiddleware` watches the message count, creates a compact summary once the trigger is reached, and keeps a smaller set of recent messages. A checkpointer stores the conversation by thread ID.


In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

conversation_agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:openai/gpt-oss-120b",
            trigger=("messages", 10),
            keep=("messages", 4),
        )
    ],
)

# A thread ID tells the checkpointer which conversation to continue.
conversation_config = {"configurable": {"thread_id": "lesson-5-demo"}}


In [ ]:
questions = [
    "What is a token?",
    "Why do input tokens matter?",
    "What is a prompt template?",
    "What does structured output give an application?",
    "Remind me why conversation history needs managing.",
]

for question in questions:
    # Send only the new message. The checkpointer restores earlier history for this thread.
    agent_result = conversation_agent.invoke(
        {"messages": [HumanMessage(content=question)]},
        config=conversation_config,
    )
    print(agent_result["messages"][-1].content)
